In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from sklearn.model_selection import cross_val_score

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 5GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/nlp-getting-started/sample_submission.csv
/kaggle/input/nlp-getting-started/test.csv
/kaggle/input/nlp-getting-started/train.csv


In [2]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.naive_bayes import GaussianNB
from sklearn.naive_bayes import BernoulliNB
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression

In [3]:
train = pd.read_csv("/kaggle/input/nlp-getting-started/train.csv")
train.head()

,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1


In [4]:
test = pd.read_csv("/kaggle/input/nlp-getting-started/test.csv")
test.head()

,id,keyword,location,text
0,0,NaN,NaN,Just happened a terrible car crash
1,2,NaN,NaN,"Heard about #earthquake is different cities, s..."
2,3,NaN,NaN,"there is a forest fire at spot pond, geese are..."
3,9,NaN,NaN,Apocalypse lighting. #Spokane #wildfires
4,11,NaN,NaN,Typhoon Soudelor kills 28 in China and Taiwan


In [5]:
print(train.shape)
print(test.shape)

(7613, 5)
(3263, 4)


In [6]:
train.isna().sum()

id             0
keyword       61
location    2533
text           0
target         0
dtype: int64

In [7]:
loc_na = train[train["location"].isnull()]
train.dropna(inplace=True,how="any")
train = train.reset_index()
train.isna().sum()

index       0
id          0
keyword     0
location    0
text        0
target      0
dtype: int64

In [8]:
print(test.keyword.mode())
print(test.location.mode())

0    deluged
dtype: object
0    New York
dtype: object


In [9]:
test["keyword"] = test.keyword.fillna("deluged")
test["location"] = test.location.fillna("New York")
test.isna().sum()

id          0
keyword     0
location    0
text        0
dtype: int64

In [10]:
from sklearn.preprocessing import LabelEncoder #convert object to int
le = LabelEncoder()

In [11]:
train["keyword"] = le.fit_transform(train["keyword"])
train["location"] = le.fit_transform(train["location"])

In [12]:
test["keyword"] = le.fit_transform(test["keyword"])
test["location"] = le.fit_transform(test["location"])

In [13]:
import spacy
nlp = spacy.load('en')

In [14]:
import re
def normalize(msg):
    
    msg = str(msg)
    msg = re.sub(r"http\S+", "", msg) #remove urls
    msg = re.sub('#[^\s]+','',msg) #remove hashtags
    msg = re.sub('@[^\s]+','',msg) #remove tags
    msg = re.sub(r'[0-9]+','', msg)
    doc = nlp(msg)
    res=[]
    for token in doc:
        if(token.is_stop or token.is_punct): #word filteration
            pass
        else:
            res.append(token.lemma_.lower())
    return " ".join(res)

In [15]:
train["text"] = train["text"].apply(normalize)

In [16]:
test["text"] = test["text"].apply(normalize)

In [17]:
text = pd.concat([train["text"],test["text"]])
text

0                                wholesale markets ablaze
1                                      try bring heavy   
2                  break news nigeria flag set ablaze aba
3                                          cry set ablaze
4                              plus look sky night ablaze
                              ...                        
3258    earthquake safety los angeles ûò safety faste...
3259    storm ri bad hurricane city&amp;other hardest ...
3260                        green line derailment chicago
3261              meg issue hazardous weather outlook hwo
3262                    activate municipal emergency plan
Name: text, Length: 8343, dtype: object

In [18]:
print(train.shape)
print(test.shape)

(5080, 6)
(3263, 4)


In [19]:
from sklearn.feature_extraction.text import TfidfVectorizer #vectorize the string
c = TfidfVectorizer(ngram_range=(1,2))
mat=pd.DataFrame(c.fit_transform(text).toarray(),columns=c.get_feature_names(),index=None)
mat

,_ius,_ius manifestation,aa,aa mgm,aa near,aaaa,aaaa ok,aaaaaaallll,aaaaaaallll ûªm,aaaaaand,...,ûókaiserjaegers,ûókaiserjaegers wiped,ûókill,ûókill rock,ûónegligence,ûónegligence fireworks,ûówe,ûówe work,ûówere,ûówere bad
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8338,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8339,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8340,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8341,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [20]:
train_ = train.drop(["id","text"],axis=1) 
train_ = pd.concat([train_, mat],axis=1)

In [21]:
train_.dropna(how = "any", inplace=True)

In [22]:
train_

,index,keyword,location,target,_ius,_ius manifestation,aa,aa mgm,aa near,aaaa,...,ûókaiserjaegers,ûókaiserjaegers wiped,ûókill,ûókill rock,ûónegligence,ûónegligence fireworks,ûówe,ûówe work,ûówere,ûówere bad
0,31.0,0.0,453.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,32.0,0.0,922.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,33.0,0.0,209.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,34.0,0.0,2054.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,35.0,0.0,1516.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5075,7575.0,220.0,2473.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5076,7577.0,220.0,50.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5077,7579.0,220.0,2703.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5078,7580.0,220.0,1507.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [23]:
train_["target"].iloc[:,0]

0       1.0
1       0.0
2       1.0
3       0.0
4       0.0
       ... 
5075    0.0
5076    0.0
5077    0.0
5078    0.0
5079    0.0
Name: target, Length: 5080, dtype: float64

In [24]:
cross_val_score(LogisticRegression(), train_.drop("target", axis=1),train_["target"].iloc[:,0],cv=10).mean()

0.5625984251968503

In [25]:
lr = LogisticRegression()
lr.fit(train_.drop(["target" ,"id"], axis=1),train_["target"].iloc[:,0])

/opt/conda/lib/python3.7/site-packages/sklearn/linear_model/_logistic.py:764: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  extra_warning_msg=_LOGISTIC_SOLVER_CONVERGENCE_MSG)


LogisticRegression()

In [26]:
test_ = test.drop(["text"],axis=1) 
test_ = pd.concat([test_, mat],axis=1)
test_

,id,keyword,location,_ius,_ius manifestation,aa,aa mgm,aa near,aaaa,aaaa ok,...,ûókaiserjaegers,ûókaiserjaegers wiped,ûókill,ûókill rock,ûónegligence,ûónegligence fireworks,ûówe,ûówe work,ûówere,ûówere bad
0,0.0,64.0,858.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2.0,64.0,858.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,3.0,64.0,858.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,9.0,64.0,858.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,11.0,64.0,858.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8338,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8339,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8340,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8341,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [27]:
test_.dropna(how = "any", inplace=True)
test_

,id,keyword,location,_ius,_ius manifestation,aa,aa mgm,aa near,aaaa,aaaa ok,...,ûókaiserjaegers,ûókaiserjaegers wiped,ûókill,ûókill rock,ûónegligence,ûónegligence fireworks,ûówe,ûówe work,ûówere,ûówere bad
0,0.0,64.0,858.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2.0,64.0,858.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,3.0,64.0,858.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,9.0,64.0,858.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,11.0,64.0,858.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3258,10861.0,64.0,858.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3259,10865.0,64.0,858.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3260,10868.0,64.0,858.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3261,10874.0,64.0,858.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [28]:
y = lr.predict(test_.drop("id", axis=1))

In [30]:
len(y)

3263

In [34]:
test_["target"] = y

In [47]:
sub = test_[["id", "target"]]

In [49]:
sub.columns = ["id", "villi", "target"]
sub.drop("villi", axis=1,inplace=True)

/opt/conda/lib/python3.7/site-packages/pandas/core/frame.py:4167: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  errors=errors,


In [50]:
sub.head()

,id,target
0,0.0,0.0
1,2.0,0.0
2,3.0,0.0
3,9.0,0.0
4,11.0,0.0


In [56]:
sub.dtypes

id        int64
target    int64
dtype: object

In [55]:
sub["id"] = sub["id"].astype("int")
sub["target"] = sub["target"].astype("int")

/opt/conda/lib/python3.7/site-packages/ipykernel_launcher.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  """Entry point for launching an IPython kernel.
/opt/conda/lib/python3.7/site-packages/ipykernel_launcher.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  


In [57]:
sub.to_csv("submission.csv", index = False)